# yolcu_agg.json — DB'den Yeniden Üretim (Imputation Sonrası)

**Neden bu notebook?**
`BILINMIYOR_HAT_TAMAMLA.ipynb` çalıştırıldıktan sonra `yolcu` tablosunda **43,028 kayıt** Bilinmiyor → gerçek HATKODU olarak güncellendi. Ham CSV (`yolculuk_temiz_final.csv`) hâlâ eski olduğu için `YOLCULUK_AGG_ANALIZ.ipynb` çalıştırılırsa eski Bilinmiyor 57K geri gelir.

**Çözüm:** Bu notebook DB'den okuyup yolcu_agg.json'u yeniden üretir.

## Pipeline
```
YOLCULUK_AGG_ANALIZ.ipynb    → ham yolcu_agg (CSV → JSON)
BILINMIYOR_HAT_TAMAMLA.ipynb → DB UPDATE 43K kayıt
YOLCU_AGG_DB_REBUILD.ipynb   → imputed yolcu_agg (DB → JSON)  ← BU NOTEBOOK
```

**Şema uyumu:** YOLCULUK_AGG_ANALIZ.ipynb çıktısıyla **bire bir aynı** (panel buradan okuyor).

**Beklenen:** Bilinmiyor 57K → 14,583 (kalan sefer dışı validator basışları)

In [1]:
import sqlite3
import pandas as pd
import json
import shutil
import time
from pathlib import Path

DATATHON_DIR = Path(r'c:\Users\asus\Desktop\Datathon')
PANEL_DIR    = DATATHON_DIR / 'panel_data'
DB_PATH      = PANEL_DIR / 'iett_data.db'

conn = sqlite3.connect(DB_PATH)
print(f'DB: {DB_PATH}')

DB: c:\Users\asus\Desktop\Datathon\panel_data\iett_data.db


In [2]:
# ── 1. KPI ──────────────────────────────────────────────────
t0 = time.time()

# Aktarma hariç (Normal) + dahil sayıları
kpi_df = pd.read_sql("""
    SELECT
        COUNT(*) AS toplam_dahil,
        SUM(CASE WHEN DAKTARMATIPI = 'Normal' THEN 1 ELSE 0 END) AS toplam_haric,
        COUNT(DISTINCT GUNCEL_HATKODU) AS hat_sayisi,
        COUNT(DISTINCT DURAKKODU) AS durak_sayisi,
        MIN(TARIH) AS tarih_min,
        MAX(TARIH) AS tarih_max
    FROM yolcu
""", conn)

k = kpi_df.iloc[0]
toplam_dahil = int(k['toplam_dahil'])
toplam_haric = int(k['toplam_haric'])
aktarma_pct  = round((toplam_dahil - toplam_haric) / toplam_dahil * 100, 2)
haftalik     = int(toplam_haric / 26)
gunluk       = int(toplam_haric / 181)
aylik        = int(toplam_haric / 6)

kpi = {
    'toplam_yolculuk':       toplam_haric,   # aktarma haric
    'toplam_yolculuk_dahil': toplam_dahil,
    'hat_sayisi':       int(k['hat_sayisi']),
    'durak_sayisi':     int(k['durak_sayisi']),
    'aktarma_orani_pct': aktarma_pct,
    'haftalik_ort':     haftalik,
    'gunluk_ort':       gunluk,
    'aylik_ort':        aylik,
    'tarih_baslangic':  str(k['tarih_min']),
    'tarih_bitis':      str(k['tarih_max']),
}
print(f'KPI ({time.time()-t0:.1f}sn):', kpi)

KPI (14.1sn): {'toplam_yolculuk': 4826695, 'toplam_yolculuk_dahil': 5135399, 'hat_sayisi': 843, 'durak_sayisi': 12942, 'aktarma_orani_pct': 6.01, 'haftalik_ort': 185642, 'gunluk_ort': 26666, 'aylik_ort': 804449, 'tarih_baslangic': '2025-01-01', 'tarih_bitis': '2025-06-30'}


In [ ]:
# ── 2. TOP HATLAR (Top 30) + hat_master'dan cinsi enrichment ──
t0 = time.time()
top_hatlar_df = pd.read_sql("""
    SELECT GUNCEL_HATKODU AS hat,
           MAX(GUNCEL_HATADI) AS ad,
           COUNT(*) AS yolculuk
    FROM yolcu
    WHERE DAKTARMATIPI = 'Normal'
      AND GUNCEL_HATKODU IS NOT NULL
    GROUP BY GUNCEL_HATKODU
    ORDER BY yolculuk DESC
    LIMIT 30
""", conn)

# hat_master.json'dan HATCINSI enrichment (METROBÜS/İETT/ÖHO/KOOP ayrımı için)
hat_master_path = PANEL_DIR / 'hat_master.json'
hat_cinsi_map = {}
if hat_master_path.exists():
    with open(hat_master_path, encoding='utf-8') as f:
        hm = json.load(f)
    if isinstance(hm, list):
        for h in hm:
            kod = h.get('HATKODU') or h.get('hatkodu', '')
            cinsi = h.get('hatcinsi', '') or h.get('HATCINSI', '')
            if kod and cinsi:
                hat_cinsi_map[str(kod).strip()] = str(cinsi).strip()

top_hatlar = []
for _, row in top_hatlar_df.iterrows():
    rec = row.to_dict()
    rec['cinsi'] = hat_cinsi_map.get(str(rec['hat']).strip(), 'Bilinmiyor')
    top_hatlar.append(rec)

print(f'Top hatlar ({time.time()-t0:.1f}sn): {len(top_hatlar)} hat (cinsi enrichment: {len(hat_cinsi_map)} hat mapping)')
print('İlk 5:')
for h in top_hatlar[:5]:
    print(f'  {h["hat"]:10s} {h["yolculuk"]:>10,} [{h["cinsi"]}] - {h.get("ad","")[:40]}')

In [4]:
# ── 3. TOP DURAKLAR (transfer agirlıklı, top 30) ──
t0 = time.time()
top_duraklar_df = pd.read_sql("""
    SELECT DURAKKODU AS durak,
           MAX(DURAKADI) AS ad,
           COUNT(*) AS yolculuk
    FROM yolcu
    WHERE DURAKKODU IS NOT NULL AND DURAKKODU > 0
    GROUP BY DURAKKODU
    ORDER BY yolculuk DESC
    LIMIT 30
""", conn)
top_duraklar = top_duraklar_df.to_dict('records')
print(f'Top duraklar ({time.time()-t0:.1f}sn): {len(top_duraklar)}')

Top duraklar (23.2sn): 30


In [5]:
# ── 4. İLÇE + MAHALLE ──
t0 = time.time()

ilce_df = pd.read_sql("""
    SELECT GUNCEL_HATILCE AS ilce, COUNT(*) AS yolculuk
    FROM yolcu
    WHERE DAKTARMATIPI = 'Normal'
      AND GUNCEL_HATILCE IS NOT NULL AND GUNCEL_HATILCE != ''
    GROUP BY GUNCEL_HATILCE
    ORDER BY yolculuk DESC
    LIMIT 40
""", conn)
ilce_bazli = ilce_df.to_dict('records')

mahalle_df = pd.read_sql("""
    SELECT MAHALLE AS mahalle, COUNT(*) AS yolculuk
    FROM yolcu
    WHERE MAHALLE IS NOT NULL AND MAHALLE != '' AND MAHALLE != 'Bilinmiyor'
    GROUP BY MAHALLE
    ORDER BY yolculuk DESC
    LIMIT 20
""", conn)
mahalle_top20 = mahalle_df.to_dict('records')

print(f'İlçe ({len(ilce_bazli)}) + Mahalle ({len(mahalle_top20)}) ({time.time()-t0:.1f}sn)')

İlçe (40) + Mahalle (20) (31.9sn)


In [6]:
# ── 5. ZAMAN BAZLI: SAAT + GÜN + AY ──
t0 = time.time()

saat_df = pd.read_sql("""
    SELECT CAST(strftime('%H', GECISZAMANI) AS INTEGER) AS saat, COUNT(*) AS yolculuk
    FROM yolcu
    WHERE DAKTARMATIPI = 'Normal' AND GECISZAMANI IS NOT NULL
    GROUP BY saat ORDER BY saat
""", conn)
saat_dagilim = saat_df.to_dict('records')

gun_df = pd.read_sql("""
    SELECT GUN AS gun, COUNT(*) AS yolculuk
    FROM yolcu WHERE DAKTARMATIPI = 'Normal' AND GUN IS NOT NULL
    GROUP BY GUN ORDER BY GUN
""", conn)
gun_dagilim = gun_df.to_dict('records')

ay_df = pd.read_sql("""
    SELECT AY AS ay, COUNT(*) AS yolculuk
    FROM yolcu WHERE DAKTARMATIPI = 'Normal' AND AY IS NOT NULL
    GROUP BY AY ORDER BY AY
""", conn)
ay_dagilim = ay_df.to_dict('records')

print(f'Zaman ({time.time()-t0:.1f}sn): saat={len(saat_dagilim)} gun={len(gun_dagilim)} ay={len(ay_dagilim)}')

Zaman (45.5sn): saat=24 gun=31 ay=6


In [ ]:
# ── 6. BİLET + GEÇİŞ + AKTARMA + ULAŞIM + OPERATÖR ──
t0 = time.time()

def gb(col, alias, limit=20, include_aktarma=False):
    """include_aktarma=False → aktarma haric (Normal) | True → aktarma dahil (DAKTARMATIPI dağılımı için)"""
    aktarma_filter = '' if include_aktarma else "AND DAKTARMATIPI = 'Normal'"
    df = pd.read_sql(f"""
        SELECT {col} AS {alias}, COUNT(*) AS sayi
        FROM yolcu
        WHERE {col} IS NOT NULL AND {col} != '' AND {col} != 'Bilinmiyor'
          {aktarma_filter}
        GROUP BY {col} ORDER BY sayi DESC LIMIT {limit}
    """, conn)
    return df.to_dict('records')

bilet_tipi      = gb('DBILETTIPI',       'tip',       10)
bilet_kategori  = gb('DBILETKATEGORI',   'kategori',  20)
gecis_turu      = gb('DGECISTURU',       'tur',       14)
# aktarma_tipi: DAKTARMATIPI dağılımı bizzat metric → include_aktarma=True ki Normal+Aktarma birlikte gelsin
aktarma_tipi    = gb('DAKTARMATIPI',     'tip',       5, include_aktarma=True)
ulasim_turu     = gb('ULASIMTURU_TANIM', 'tur',       5)
operator        = gb('USTOPERATORADI',   'operator',  10)

# aktarma_tipi'ye pct alanı ekle (panel KPI'ı için)
_at_total = sum(r['sayi'] for r in aktarma_tipi)
for r in aktarma_tipi:
    r['pct'] = round(r['sayi'] / _at_total * 100, 2) if _at_total else 0

print(f'Bilet/Gecis/Aktarma ({time.time()-t0:.1f}sn)')
print(f'  bilet_tipi: {len(bilet_tipi)}, bilet_kategori: {len(bilet_kategori)}, gecis: {len(gecis_turu)}')
print(f'  aktarma_tipi: {[(r["tip"], r["sayi"], f"%{r[\"pct\"]}") for r in aktarma_tipi]}')

In [ ]:
# ── 7. ARAÇ BAZLI: GARAJ + MARKA + CİNS + EMISYON + YAKIT ──
# NOT: yolcu tablosundaki MARKA alanı OPERATÖR adı (AKIA, KARSAN gibi firmalar).
# Gerçek araç markası MODEL alanının ilk kelimesinde (MERCEDES CONECTO, BMC PROCITY 285).
# %93.9 MARKA-MODEL uyumsuz → MARKA yerine MODEL'den imputation.
# DİKKAT: SQLite alias shadowing bug — GROUP BY 'marka' alias'i MARKA orijinal kolonuyla
# çakışıyor. Çözüm: SUBQUERY ile önce computed kolon, sonra GROUP BY.
t0 = time.time()

garaj_bazli     = gb('GARAJ',     'garaj',   15)
arac_cinsi      = gb('ARACCINSI', 'cinsi',   10)
emisyon_bazli   = gb('EMISYON',   'emisyon', 5)
yakit_bazli     = gb('YAKITTURU', 'yakit',   5)

# MARKA — MODEL ilk kelimesinden gerçek araç markası (SUBQUERY ile alias shadowing engellendi)
marka_df = pd.read_sql("""
    SELECT marka_temiz AS marka, COUNT(*) AS sayi FROM (
        SELECT
            CASE
                WHEN MODEL LIKE 'BMC%' OR MODEL = 'BMC' OR MODEL LIKE 'PROCITY%'
                  OR MODEL LIKE 'BELDE%'                                               THEN 'BMC'
                WHEN MODEL LIKE 'OTOKAR%' OR MODEL = 'OTOKAR' OR MODEL LIKE 'KENT%'
                  OR MODEL LIKE 'SULTAN%' OR MODEL LIKE 'NEOCITY%'                     THEN 'OTOKAR'
                WHEN MODEL LIKE 'MERCEDES%' OR MODEL LIKE 'CONECTO%'
                  OR MODEL LIKE 'CITARO%' OR MODEL LIKE 'CAPACITY%'                    THEN 'MERCEDES'
                WHEN MODEL LIKE 'KARSAN%' OR MODEL = 'KARSAN' OR MODEL LIKE 'AVENUE%'
                  OR MODEL LIKE 'AVANCITY%' OR MODEL LIKE 'CITYPORT%'                  THEN 'KARSAN'
                WHEN MODEL LIKE 'TEMSA%' OR MODEL = 'TEMSA'                            THEN 'TEMSA'
                WHEN MODEL LIKE 'GÜLERYÜZ%' OR MODEL LIKE 'COBRA%'                     THEN 'GÜLERYÜZ'
                WHEN MODEL LIKE 'ANADOLU%' OR MODEL LIKE 'ISUZU%'
                  OR MODEL LIKE 'CITIPORT%'                                            THEN 'ISUZU'
                WHEN MODEL LIKE 'AKIA%' OR MODEL = 'AKIA' OR MODEL LIKE 'ULTRA%'
                  OR MODEL LIKE 'LF25%'                                                THEN 'AKIA'
                WHEN MODEL LIKE 'TEZELLER%'                                            THEN 'TEZELLER'
                WHEN MODEL LIKE 'MAN%' OR MODEL = 'MAN'                                THEN 'MAN'
                WHEN MODEL LIKE 'BREDA%'                                               THEN 'BREDAMENARINI'
                WHEN MODEL LIKE 'CLEANVAC%' OR MODEL = 'CLEANVAC'                      THEN 'CLEANVAC'
                WHEN MODEL LIKE 'GREEN%'                                               THEN 'GREEN CAR'
                WHEN MODEL LIKE 'SGMS%'                                                THEN 'SGMS'
                WHEN MODEL LIKE 'PİLOTCAR%' OR MODEL LIKE 'PILOTCAR%'                  THEN 'PİLOTCAR'
                ELSE 'Diğer'
            END AS marka_temiz
        FROM yolcu
        WHERE DAKTARMATIPI = 'Normal'
          AND MODEL IS NOT NULL AND MODEL != '' AND MODEL != 'Bilinmiyor'
    ) WHERE marka_temiz != 'Diğer'
    GROUP BY marka_temiz
    ORDER BY sayi DESC
    LIMIT 10
""", conn)
marka_bazli = marka_df.to_dict('records')

print(f'Araç bazlı ({time.time()-t0:.1f}sn)')
print(f'  garaj: {len(garaj_bazli)}, marka: {len(marka_bazli)}, cinsi: {len(arac_cinsi)}')
print(f'  MARKA (MODEL\'den imputed, subquery ile):')
toplam = sum(m["sayi"] for m in marka_bazli)
for m in marka_bazli:
    pct = m["sayi"]/toplam*100 if toplam else 0
    print(f'    {m["marka"]:15s} {m["sayi"]:>10,} (%{pct:.1f})')

In [ ]:
# ── 8. JSON ÜRET — yolcu_agg.json şeması ──
yolcu_agg = {
    'kpi':            kpi,
    'top_hatlar':     top_hatlar,         # cinsi alanı dahil (hat_master.json enrichment)
    'top_duraklar':   top_duraklar,
    'ilce_bazli':     ilce_bazli,
    'mahalle_top20':  mahalle_top20,
    'saat_dagilim':   saat_dagilim,
    'gun_dagilim':    gun_dagilim,
    'ay_dagilim':     ay_dagilim,
    'bilet_tipi':     bilet_tipi,
    'bilet_kategori': bilet_kategori,
    'gecis_turu':     gecis_turu,
    'aktarma_tipi':   aktarma_tipi,       # pct alanı dahil
    'ulasim_turu':    ulasim_turu,
    'operator':       operator,
    'garaj_bazli':    garaj_bazli,
    'marka_bazli':    marka_bazli,
    'arac_cinsi':     arac_cinsi,
    'emisyon_bazli':  emisyon_bazli,
    'yakit_bazli':    yakit_bazli,
    'aktarma_sure_dagilim': aktarma_sure_dagilim,  # 5 grup (0-5/5-10/10-20/20-40/40+)
    'haftaici_haftasonu':   haftaici_haftasonu,    # 2 dönem + gunluk_ort + pct
    'ilce_bilet_matris':    ilce_bilet_matris,     # YENİ: 12 ilçe × 7 kategori sosyal profil
    'meta': {
        'kaynak':         'iett_data.db yolcu tablosu (DB-rebuild)',
        'tarih':          '2025 H1',
        'aktarma_haric':  True,
        'imputation':     'BILINMIYOR_HAT_TAMAMLA.ipynb sonrası (43K kayıt atandı)',
        'olusturulma':    pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
    },
}

print(f'JSON şeması hazır. Top hat: {yolcu_agg["top_hatlar"][0]["hat"]} ({yolcu_agg["top_hatlar"][0]["yolculuk"]:,} yolcu, cinsi: {yolcu_agg["top_hatlar"][0].get("cinsi","?")})')
print(f'Toplam kategori: {len([k for k in yolcu_agg if k != "meta"])}')

In [ ]:
# ── 7c. HAFTA İÇİ / HAFTA SONU DAĞILIMI ──
# strftime('%w'): 0=Pazar, 1=Pzt, ..., 6=Cmt → Cmt+Pazar = hafta sonu
# ŞEMA: panel adapter 'tur' (Hafta Ici/Hafta Sonu), 'yolculuk', 'gun_sayisi', 'gunluk_ort' bekliyor.
t0 = time.time()

hi_hs_df = pd.read_sql("""
    SELECT
        CASE WHEN CAST(strftime('%w', TARIH) AS INTEGER) IN (0, 6)
             THEN 'Hafta Sonu' ELSE 'Hafta Ici' END AS tur,
        COUNT(*) AS yolculuk,
        COUNT(DISTINCT TARIH) AS gun_sayisi
    FROM yolcu
    WHERE DAKTARMATIPI = 'Normal' AND TARIH IS NOT NULL
    GROUP BY tur
""", conn)

haftaici_haftasonu = []
for _, row in hi_hs_df.iterrows():
    rec = row.to_dict()
    rec['gunluk_ort'] = int(rec['yolculuk'] / rec['gun_sayisi']) if rec['gun_sayisi'] else 0
    haftaici_haftasonu.append(rec)

# Sıra: Hafta Ici önce
haftaici_haftasonu.sort(key=lambda x: 0 if 'Ici' in x['tur'] else 1)

_hh_total = sum(r['yolculuk'] for r in haftaici_haftasonu)
print(f'Hafta İçi/Sonu ({time.time()-t0:.1f}sn) — toplam {_hh_total:,}:')
for r in haftaici_haftasonu:
    pct = round(r['yolculuk'] / _hh_total * 100, 2) if _hh_total else 0
    print(f'  {r["tur"]:11s} {r["yolculuk"]:>10,} ({r["gun_sayisi"]} gün, günlük ort {r["gunluk_ort"]:,}) %{pct}')

In [ ]:
# ── 7d. İLÇE × BİLET KATEGORİ MATRİSİ (Sosyal Profil Heatmap için) ──
# Hangi ilçede hangi sosyal grup ağırlıkta? Üniversite ilçeleri vs yaşlı yoğun ilçeler vs.
# Top 12 ilçe × Top 7 kategori — stacked bar chart kaynağı.
t0 = time.time()

# Top 12 ilçe (aktarma haric)
ilce_top = pd.read_sql("""
    SELECT GUNCEL_HATILCE AS ilce, COUNT(*) AS toplam
    FROM yolcu
    WHERE DAKTARMATIPI = 'Normal'
      AND GUNCEL_HATILCE IS NOT NULL AND GUNCEL_HATILCE != ''
      AND GUNCEL_HATILCE != 'Bilinmiyor'
    GROUP BY GUNCEL_HATILCE ORDER BY toplam DESC LIMIT 12
""", conn)
top_ilceler_list = ilce_top['ilce'].tolist()

# Top 7 bilet kategori
kat_top = pd.read_sql("""
    SELECT DBILETKATEGORI AS kat, COUNT(*) AS s
    FROM yolcu
    WHERE DAKTARMATIPI = 'Normal'
      AND DBILETKATEGORI IS NOT NULL AND DBILETKATEGORI != '' AND DBILETKATEGORI != 'Bilinmiyor'
    GROUP BY DBILETKATEGORI ORDER BY s DESC LIMIT 7
""", conn)
top_kategoriler_list = kat_top['kat'].tolist()

# Matris (tek SQL)
ph_i = ','.join(['?'] * len(top_ilceler_list))
ph_k = ','.join(['?'] * len(top_kategoriler_list))
mat_raw = pd.read_sql(f"""
    SELECT GUNCEL_HATILCE AS ilce, DBILETKATEGORI AS kat, COUNT(*) AS s
    FROM yolcu
    WHERE DAKTARMATIPI = 'Normal'
      AND GUNCEL_HATILCE IN ({ph_i})
      AND DBILETKATEGORI IN ({ph_k})
    GROUP BY GUNCEL_HATILCE, DBILETKATEGORI
""", conn, params=top_ilceler_list + top_kategoriler_list)

# Pivot → list of dicts
piv = {}
for _, r in mat_raw.iterrows():
    piv.setdefault(r['ilce'], {})[r['kat']] = int(r['s'])

ilce_bilet_matris_rows = []
for ilce in top_ilceler_list:
    row = {'ilce': ilce, 'toplam': sum(piv.get(ilce, {}).values())}
    for kat in top_kategoriler_list:
        row[kat] = piv.get(ilce, {}).get(kat, 0)
    ilce_bilet_matris_rows.append(row)

ilce_bilet_matris = {
    'ilceler':     top_ilceler_list,
    'kategoriler': top_kategoriler_list,
    'matris':      ilce_bilet_matris_rows,
}

print(f'İlçe × Bilet Kategori matrisi ({time.time()-t0:.1f}sn):')
print(f'  {len(top_ilceler_list)} ilçe × {len(top_kategoriler_list)} kategori')
print(f'  En öğrenci ağırlıklı ilk 3:')
sorted_by_ogr = sorted(ilce_bilet_matris_rows, key=lambda r: r.get('Öğrenci',0)/max(r.get('toplam',1),1), reverse=True)[:3]
for r in sorted_by_ogr:
    pct = r.get('Öğrenci',0)/max(r.get('toplam',1),1)*100
    print(f'    {r["ilce"]:18s} öğrenci %{pct:.1f} ({r.get("Öğrenci",0):,} / {r["toplam"]:,})')

In [ ]:
# ── 10. DOĞRULAMA ──
with open(OUT_D, encoding='utf-8') as f:
    chk = json.load(f)

# Bilinmiyor oranı kontrol — top_hatlar'da Bilinmiyor var mı?
bilinmiyor_top = sum(1 for h in chk['top_hatlar'] if h.get('hat') == 'Bilinmiyor')
bilinmiyor_yolculuk = next((h['yolculuk'] for h in chk['top_hatlar'] if h.get('hat') == 'Bilinmiyor'), 0)

print('=== DOĞRULAMA ===')
print(f'Toplam yolculuk (aktarma haric): {chk["kpi"]["toplam_yolculuk"]:,}')
print(f'Hat sayisi: {chk["kpi"]["hat_sayisi"]:,}')
print(f'Durak sayisi: {chk["kpi"]["durak_sayisi"]:,}')
print(f'Aktarma oranı: %{chk["kpi"]["aktarma_orani_pct"]}')
print()
print(f'Top hatlarda Bilinmiyor: {bilinmiyor_yolculuk:,} yolculuk' if bilinmiyor_top else 'Bilinmiyor top 30 disinda ✓')
print(f'Top 5 hat (cinsi enrichment ile):')
for h in chk['top_hatlar'][:5]:
    print(f'  {h["hat"]:8s} {h["yolculuk"]:>10,}  [{h.get("cinsi","?"):10s}]')

print()
print(f'Saat dagilimi: {len(chk["saat_dagilim"])} (24 olmali)')
print(f'Ay dagilimi:   {len(chk["ay_dagilim"])} (6 olmali)')
print(f'İlçe dagilimi: {len(chk["ilce_bazli"])}')

# YENİ ALANLAR DOĞRULAMASI (panel adapter şeması)
print()
print('=== YENİ EKLENEN ALANLAR ===')
print(f'aktarma_tipi pct alanı: {all("pct" in r for r in chk["aktarma_tipi"])}')
for r in chk['aktarma_tipi']:
    print(f'  {r["tip"]:12s} {r["sayi"]:>10,}  %{r.get("pct","?")}')

print()
print(f'aktarma_sure_dagilim ({len(chk.get("aktarma_sure_dagilim", []))} grup) [şema: grup/adet]:')
for r in chk.get('aktarma_sure_dagilim', []):
    print(f'  {r.get("grup","?"):10s} {r.get("adet",0):>8,}')

print()
print(f'haftaici_haftasonu ({len(chk.get("haftaici_haftasonu", []))} dönem) [şema: tur/yolculuk/gun_sayisi/gunluk_ort]:')
for r in chk.get('haftaici_haftasonu', []):
    print(f'  {r.get("tur","?"):11s} {r.get("yolculuk",0):>10,} ({r.get("gun_sayisi",0)} gün, günlük {r.get("gunluk_ort",0):,})')

print()
print('Meta:', chk['meta'])

In [10]:
# ── 9. YAZIM — Backup + Panel kopya ──
OUT_D = PANEL_DIR / 'yolcu_agg.json'
OUT_P = Path(r'c:\Users\asus\Desktop\iett_panel\panel_data\yolcu_agg.json')
BK    = PANEL_DIR / '_arsiv_v5'
BK.mkdir(exist_ok=True)

# Backup eski JSON
if OUT_D.exists():
    bk_path = BK / f'yolcu_agg_pre_rebuild_{pd.Timestamp.now().strftime("%Y%m%d_%H%M")}.json'
    shutil.copy2(OUT_D, bk_path)
    print(f'Backup: {bk_path}')

with open(OUT_D, 'w', encoding='utf-8') as f:
    json.dump(yolcu_agg, f, ensure_ascii=False, indent=2)
print(f'Datathon: {OUT_D} ({OUT_D.stat().st_size/1024:.0f} KB)')

if OUT_P.parent.exists():
    shutil.copy2(OUT_D, OUT_P)
    print(f'Panel:    {OUT_P}')

conn.close()
print('DB kapatildi.')

Backup: c:\Users\asus\Desktop\Datathon\panel_data\_arsiv_v5\yolcu_agg_pre_rebuild_20260515_1458.json
Datathon: c:\Users\asus\Desktop\Datathon\panel_data\yolcu_agg.json (20 KB)
Panel:    c:\Users\asus\Desktop\iett_panel\panel_data\yolcu_agg.json
DB kapatildi.


In [ ]:
# ── 11. filo_yolcu_agg.json — MARKA dagilimi imputation patch ──
# Bu JSON Yolcu/Filo/Bilet 3 sekmenin "marka_dagilim" kaynağı (operatör adı içeriyordu).
# YOLCULUK_AGG_ANALIZ.ipynb (CSV tabanlı, dokunulmaz) tarafından üretiliyor → sadece marka_dagilim
# alanını DB'den re-impute ile güncelleyelim.
# NOT: Önceki cell conn.close() yaptı, burada yeniden açıyoruz.

import time as _time
t0 = _time.time()
conn2 = sqlite3.connect(DB_PATH)

# Datathon ve Panel yolu
FYA_PATHS = [
    Path(r'c:\Users\asus\Desktop\Datathon\panel_data\filo_yolcu_agg.json'),
    Path(r'c:\Users\asus\Desktop\iett_panel\panel_data\filo_yolcu_agg.json'),
]

# DB'den imputed marka_dagilim üret (cell-8'deki SUBQUERY ile aynı)
fya_marka_df = pd.read_sql("""
    SELECT marka_temiz AS marka, COUNT(*) AS yolculuk FROM (
        SELECT
            CASE
                WHEN MODEL LIKE 'BMC%' OR MODEL = 'BMC' OR MODEL LIKE 'PROCITY%'
                  OR MODEL LIKE 'BELDE%'                                               THEN 'BMC'
                WHEN MODEL LIKE 'OTOKAR%' OR MODEL = 'OTOKAR' OR MODEL LIKE 'KENT%'
                  OR MODEL LIKE 'SULTAN%' OR MODEL LIKE 'NEOCITY%'                     THEN 'OTOKAR'
                WHEN MODEL LIKE 'MERCEDES%' OR MODEL LIKE 'CONECTO%'
                  OR MODEL LIKE 'CITARO%' OR MODEL LIKE 'CAPACITY%'                    THEN 'MERCEDES'
                WHEN MODEL LIKE 'KARSAN%' OR MODEL = 'KARSAN' OR MODEL LIKE 'AVENUE%'
                  OR MODEL LIKE 'AVANCITY%' OR MODEL LIKE 'CITYPORT%'                  THEN 'KARSAN'
                WHEN MODEL LIKE 'TEMSA%' OR MODEL = 'TEMSA'                            THEN 'TEMSA'
                WHEN MODEL LIKE 'GÜLERYÜZ%' OR MODEL LIKE 'COBRA%'                     THEN 'GÜLERYÜZ'
                WHEN MODEL LIKE 'ANADOLU%' OR MODEL LIKE 'ISUZU%'
                  OR MODEL LIKE 'CITIPORT%'                                            THEN 'ISUZU'
                WHEN MODEL LIKE 'AKIA%' OR MODEL = 'AKIA' OR MODEL LIKE 'ULTRA%'
                  OR MODEL LIKE 'LF25%'                                                THEN 'AKIA'
                WHEN MODEL LIKE 'TEZELLER%'                                            THEN 'TEZELLER'
                WHEN MODEL LIKE 'MAN%' OR MODEL = 'MAN'                                THEN 'MAN'
                WHEN MODEL LIKE 'BREDA%'                                               THEN 'BREDAMENARINI'
                WHEN MODEL LIKE 'CLEANVAC%' OR MODEL = 'CLEANVAC'                      THEN 'CLEANVAC'
                WHEN MODEL LIKE 'GREEN%'                                               THEN 'GREEN CAR'
                WHEN MODEL LIKE 'SGMS%'                                                THEN 'SGMS'
                WHEN MODEL LIKE 'PİLOTCAR%' OR MODEL LIKE 'PILOTCAR%'                  THEN 'PİLOTCAR'
                ELSE 'Diğer'
            END AS marka_temiz
        FROM yolcu
        WHERE MODEL IS NOT NULL AND MODEL != '' AND MODEL != 'Bilinmiyor'
    ) WHERE marka_temiz != 'Diğer'
    GROUP BY marka_temiz
    ORDER BY yolculuk DESC
""", conn2)
conn2.close()

# pct alanı + format (filo_yolcu_agg şeması: marka/yolculuk/pct)
_top = int(fya_marka_df['yolculuk'].sum())
fya_marka_dagilim = []
for _, r in fya_marka_df.iterrows():
    fya_marka_dagilim.append({
        'marka':    r['marka'],
        'yolculuk': int(r['yolculuk']),
        'pct':      round(int(r['yolculuk'])/_top*100, 2) if _top else 0,
    })

print(f'Yeni marka_dagilim (filo_yolcu_agg.json için):')
for m in fya_marka_dagilim:
    print(f'  {m["marka"]:15s} {m["yolculuk"]:>10,} (%{m["pct"]})')

# Mevcut JSON'lara patch
for fya_path in FYA_PATHS:
    if not fya_path.exists():
        print(f'YOK: {fya_path}')
        continue
    with open(fya_path, encoding='utf-8') as f:
        fya = json.load(f)
    fya['marka_dagilim'] = fya_marka_dagilim
    # Meta'ya not düş
    if 'meta' in fya:
        fya['meta']['marka_dagilim_patch'] = f'MODEL imputation ({pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")})'
    with open(fya_path, 'w', encoding='utf-8') as f:
        json.dump(fya, f, ensure_ascii=False, indent=1)
    print(f'OK: {fya_path}')

print(f'({_time.time()-t0:.1f}sn)')

In [11]:
# ── 10. DOĞRULAMA ──
with open(OUT_D, encoding='utf-8') as f:
    chk = json.load(f)

# Bilinmiyor oranı kontrol — top_hatlar'da Bilinmiyor var mı?
bilinmiyor_top = sum(1 for h in chk['top_hatlar'] if h.get('hat') == 'Bilinmiyor')
bilinmiyor_yolculuk = next((h['yolculuk'] for h in chk['top_hatlar'] if h.get('hat') == 'Bilinmiyor'), 0)

print('=== DOĞRULAMA ===')
print(f'Toplam yolculuk (aktarma haric): {chk["kpi"]["toplam_yolculuk"]:,}')
print(f'Hat sayisi: {chk["kpi"]["hat_sayisi"]:,}')
print(f'Durak sayisi: {chk["kpi"]["durak_sayisi"]:,}')
print(f'Aktarma oranı: %{chk["kpi"]["aktarma_orani_pct"]}')
print()
print(f'Top hatlarda Bilinmiyor: {bilinmiyor_yolculuk:,} yolculuk' if bilinmiyor_top else 'Bilinmiyor top 30 disinda ✓')
print(f'Top 5 hat:')
for h in chk['top_hatlar'][:5]:
    print(f'  {h["hat"]:8s} {h["yolculuk"]:>10,}')

print()
print(f'Saat dagilimi: {len(chk["saat_dagilim"])} (24 olmali)')
print(f'Ay dagilimi:   {len(chk["ay_dagilim"])} (6 olmali)')
print(f'İlçe dagilimi: {len(chk["ilce_bazli"])}')
print()
print('Meta:', chk['meta'])

=== DOĞRULAMA ===
Toplam yolculuk (aktarma haric): 4,826,695
Hat sayisi: 843
Durak sayisi: 12,942
Aktarma oranı: %6.01

Bilinmiyor top 30 disinda ✓
Top 5 hat:
  34          889,207
  34A         107,795
  500T         41,435
  19F          37,622
  132M         36,354

Saat dagilimi: 24 (24 olmali)
Ay dagilimi:   6 (6 olmali)
İlçe dagilimi: 40

Meta: {'kaynak': 'iett_data.db yolcu tablosu (DB-rebuild)', 'tarih': '2025 H1', 'aktarma_haric': True, 'imputation': 'BILINMIYOR_HAT_TAMAMLA.ipynb sonrası (43K kayıt atandı)', 'olusturulma': '2026-05-15 14:58'}
